<a href="https://colab.research.google.com/github/MohamedAbdeenM7/recommendation-system/blob/student1%2Fdata-preprocessing/student1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import userdata

# Step 1: Clone the repo
%cd /content
token = userdata.get('GITHUB_TOKEN')
os.environ['TOKEN'] = token

!git clone https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git
%cd recommendation-system

# Step 2: Write ingest.py
ingest_code = '''import pandas as pd
import os

def load_ratings(path: str) -> pd.DataFrame:
    """
    Load ratings from CSV file

    Parameters:
        path: File path on Drive

    Returns:
        DataFrame with columns: user_id, product_id, rating, date
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_csv(
        path,
        header=None,
        names=["user_id", "product_id", "rating", "timestamp"]
    )

    df["date"] = pd.to_datetime(df["timestamp"], unit="s")
    df = df.drop(columns=["timestamp"])

    print(f"✅ Data loaded successfully!")
    print(f"📊 Shape: {df.shape}")
    print(f"👥 Unique Users: {df[\'user_id\'].nunique():,}")
    print(f"📦 Unique Products: {df[\'product_id\'].nunique():,}")

    return df


def validate_data(df: pd.DataFrame) -> bool:
    """
    Validate the loaded data

    Parameters:
        df: DataFrame to validate

    Returns:
        True if data is valid
    """
    required_columns = ["user_id", "product_id", "rating", "date"]

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"❌ Missing column: {col}")

    if not df["rating"].between(1, 5).all():
        raise ValueError("❌ Ratings must be between 1 and 5")

    if df.isnull().sum().sum() > 0:
        raise ValueError("❌ Data contains missing values")

    print("✅ Data validation passed!")
    return True


if __name__ == "__main__":
    DATA_PATH = "/content/drive/MyDrive/recommendation-data/Electronics.csv"
    df = load_ratings(DATA_PATH)
    validate_data(df)
    print(df.head())
'''

with open("src/data/ingest.py", "w") as f:
    f.write(ingest_code)
print("✅ ingest.py created!")

# Step 3: Push to GitHub
!git config --global user.email "MohamedAbdeenM7@gmail.com"
!git config --global user.name "Student1"
!git checkout student1/data-preprocessing
!git remote set-url origin https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git
!git add src/data/ingest.py
!git commit -m "✅ M1: Add ingest.py"
!git push origin student1/data-preprocessing

print("✅ ingest.py pushed to GitHub!")

/content
Cloning into 'recommendation-system'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 64 (delta 19), reused 38 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 131.30 KiB | 4.53 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/recommendation-system
✅ ingest.py created!
Branch 'student1/data-preprocessing' set up to track remote branch 'student1/data-preprocessing' from 'origin'.
Switched to a new branch 'student1/data-preprocessing'
[student1/data-preprocessing ced3ba6] ✅ M1: Add ingest.py
 1 file changed, 64 insertions(+)
 create mode 100644 src/data/ingest.py
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.15 KiB | 1.15 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2)

In [5]:
import os
from google.colab import userdata

# Step 1: Clone the repo
%cd /content
token = userdata.get('GITHUB_TOKEN')
os.environ['TOKEN'] = token

!git clone https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git
%cd recommendation-system

preprocess_code = '''import pandas as pd

def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate rows"""
    before = df.shape[0]
    df = df.drop_duplicates()
    after = df.shape[0]
    print(f"✅ Removed {before - after:,} duplicates")
    return df


def filter_ratings(df: pd.DataFrame) -> pd.DataFrame:
    """Filter ratings to be between 1 and 5"""
    df = df[df["rating"].between(1, 5)]
    print(f"✅ Ratings filtered: {df.shape[0]:,} rows remaining")
    return df


def filter_active_users(df: pd.DataFrame, min_ratings: int = 5) -> pd.DataFrame:
    """Keep only users with at least min_ratings ratings"""
    user_counts = df["user_id"].value_counts()
    active_users = user_counts[user_counts >= min_ratings].index
    df = df[df["user_id"].isin(active_users)]
    print(f"✅ Active users filtered: {df["user_id"].nunique():,} users remaining")
    return df


def filter_popular_products(df: pd.DataFrame, min_ratings: int = 5) -> pd.DataFrame:
    """Keep only products with at least min_ratings ratings"""
    product_counts = df["product_id"].value_counts()
    popular_products = product_counts[product_counts >= min_ratings].index
    df = df[df["product_id"].isin(popular_products)]
    print(f"✅ Popular products filtered: {df["product_id"].nunique():,} products remaining")
    return df


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full preprocessing pipeline

    Parameters:
        df: Raw DataFrame

    Returns:
        Cleaned DataFrame
    """
    print("🔄 Starting preprocessing pipeline...")
    df = remove_duplicates(df)
    df = filter_ratings(df)
    df = filter_active_users(df, min_ratings=5)
    df = filter_popular_products(df, min_ratings=5)
    print(f"✅ Preprocessing complete!")
    print(f"📊 Final Shape: {df.shape}")
    print(f"👥 Final Unique Users: {df["user_id"].nunique():,}")
    print(f"📦 Final Unique Products: {df["product_id"].nunique():,}")
    return df


def save_clean_data(df: pd.DataFrame, path: str) -> None:
    """Save cleaned data to CSV"""
    df.to_csv(path, index=False)
    print(f"✅ Clean data saved to: {path}")


if __name__ == "__main__":
    from ingest import load_ratings
    DATA_PATH = "/content/drive/MyDrive/recommendation-data/Electronics.csv"
    SAVE_PATH = "/content/drive/MyDrive/recommendation-data/ratings_clean.csv"
    df = load_ratings(DATA_PATH)
    df = preprocess(df)
    save_clean_data(df, SAVE_PATH)
'''

with open("src/data/preprocess.py", "w") as f:
    f.write(preprocess_code)
print("✅ preprocess.py created!")

# Push to GitHub
!git add src/data/preprocess.py
!git commit -m "✅ M1: Add preprocess.py"
!git push origin student1/data-preprocessing

print("✅ preprocess.py pushed to GitHub!")

/content
fatal: destination path 'recommendation-system' already exists and is not an empty directory.
/content/recommendation-system
✅ preprocess.py created!
On branch student1/data-preprocessing
Your branch is up to date with 'origin/student1/data-preprocessing'.

nothing to commit, working tree clean
Everything up-to-date
✅ preprocess.py pushed to GitHub!


In [1]:
import os
from google.colab import userdata

# Step 1: Clone the repo
%cd /content
token = userdata.get('GITHUB_TOKEN')
os.environ['TOKEN'] = token

!git clone https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git
%cd recommendation-system


test_code = '''import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "src", "data")))

from preprocess import (
    remove_duplicates,
    filter_ratings,
    filter_active_users,
    filter_popular_products,
    preprocess
)

# ============================================================
# Sample Test Data
# ============================================================
def get_sample_data():
    return pd.DataFrame({
        "user_id":    ["u1", "u1", "u1", "u1", "u1", "u2", "u2", "u2", "u2", "u2", "u1"],
        "product_id": ["p1", "p2", "p3", "p4", "p5", "p1", "p2", "p3", "p4", "p5", "p1"],
        "rating":     [5,    4,    3,    2,    1,    4,    3,    2,    1,    5,    5  ],
        "date":       pd.to_datetime(["2021-01-01"] * 11)
    })

# ============================================================
# Tests
# ============================================================
def test_remove_duplicates():
    df = get_sample_data()
    result = remove_duplicates(df)
    assert result.duplicated().sum() == 0
    print("✅ test_remove_duplicates passed!")

def test_filter_ratings():
    df = get_sample_data()
    df.loc[0, "rating"] = 6
    df.loc[1, "rating"] = 0
    result = filter_ratings(df)
    assert result["rating"].between(1, 5).all()
    print("✅ test_filter_ratings passed!")

def test_filter_active_users():
    df = get_sample_data()
    df = remove_duplicates(df)
    result = filter_active_users(df, min_ratings=5)
    user_counts = result["user_id"].value_counts()
    assert (user_counts >= 5).all()
    print("✅ test_filter_active_users passed!")

def test_filter_popular_products():
    df = get_sample_data()
    df = remove_duplicates(df)
    result = filter_popular_products(df, min_ratings=2)
    product_counts = result["product_id"].value_counts()
    assert (product_counts >= 2).all()
    print("✅ test_filter_popular_products passed!")

def test_preprocess():
    df = get_sample_data()
    result = preprocess(df)
    assert result.duplicated().sum() == 0
    assert result["rating"].between(1, 5).all()
    assert result.isnull().sum().sum() == 0
    print("✅ test_preprocess passed!")

# ============================================================
# Run All Tests
# ============================================================
if __name__ == "__main__":
    print("🔄 Running all tests...\\n")
    test_remove_duplicates()
    test_filter_ratings()
    test_filter_active_users()
    test_filter_popular_products()
    test_preprocess()
    print("\\n🎉 All tests passed!")
'''

with open("tests/test_preprocess.py", "w") as f:
    f.write(test_code)
print("✅ test_preprocess.py created!")

# Push to GitHub
!git add tests/test_preprocess.py
!git commit -m "✅ M1: Add test_preprocess.py"
!git push origin student1/data-preprocessing

print("✅ test_preprocess.py pushed to GitHub!")

/content
fatal: destination path 'recommendation-system' already exists and is not an empty directory.
/content/recommendation-system
✅ test_preprocess.py created!
[main 739aded] ✅ M1: Add test_preprocess.py
Everything up-to-date
✅ test_preprocess.py pushed to GitHub!


In [2]:
import os
from google.colab import userdata

%cd /content/recommendation-system

token = userdata.get('GITHUB_TOKEN')
os.environ['TOKEN'] = token

test_code = '''import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "src", "data")))

from preprocess import (
    remove_duplicates,
    filter_ratings,
    filter_active_users,
    filter_popular_products,
    preprocess
)

def get_sample_data():
    return pd.DataFrame({
        "user_id":    ["u1", "u1", "u1", "u1", "u1", "u2", "u2", "u2", "u2", "u2", "u1"],
        "product_id": ["p1", "p2", "p3", "p4", "p5", "p1", "p2", "p3", "p4", "p5", "p1"],
        "rating":     [5,    4,    3,    2,    1,    4,    3,    2,    1,    5,    5  ],
        "date":       pd.to_datetime(["2021-01-01"] * 11)
    })

def test_remove_duplicates():
    df = get_sample_data()
    result = remove_duplicates(df)
    assert result.duplicated().sum() == 0
    print("✅ test_remove_duplicates passed!")

def test_filter_ratings():
    df = get_sample_data()
    df.loc[0, "rating"] = 6
    df.loc[1, "rating"] = 0
    result = filter_ratings(df)
    assert result["rating"].between(1, 5).all()
    print("✅ test_filter_ratings passed!")

def test_filter_active_users():
    df = get_sample_data()
    df = remove_duplicates(df)
    result = filter_active_users(df, min_ratings=5)
    user_counts = result["user_id"].value_counts()
    assert (user_counts >= 5).all()
    print("✅ test_filter_active_users passed!")

def test_filter_popular_products():
    df = get_sample_data()
    df = remove_duplicates(df)
    result = filter_popular_products(df, min_ratings=2)
    product_counts = result["product_id"].value_counts()
    assert (product_counts >= 2).all()
    print("✅ test_filter_popular_products passed!")

def test_preprocess():
    df = get_sample_data()
    result = preprocess(df)
    assert result.duplicated().sum() == 0
    assert result["rating"].between(1, 5).all()
    assert result.isnull().sum().sum() == 0
    print("✅ test_preprocess passed!")

if __name__ == "__main__":
    print("🔄 Running all tests...\\n")
    test_remove_duplicates()
    test_filter_ratings()
    test_filter_active_users()
    test_filter_popular_products()
    test_preprocess()
    print("\\n🎉 All tests passed!")
'''

with open("tests/test_preprocess.py", "w") as f:
    f.write(test_code)
print("✅ test_preprocess.py created!")

!git config --global user.email "MohamedAbdeenM7@gmail.com"
!git config --global user.name "Student1"
!git remote set-url origin https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git
!git add tests/test_preprocess.py
!git commit -m "✅ M1: Add test_preprocess.py"
!git push origin main

print("✅ test_preprocess.py pushed to GitHub!")

/content/recommendation-system
✅ test_preprocess.py created!
[main b3363df] ✅ M1: Add test_preprocess.py
 1 file changed, 9 deletions(-)
To https://github.com/MohamedAbdeenM7/recommendation-system.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/MohamedAbdeenM7/recommendation-system.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.
✅ test_preprocess.py pushed to GitHub!


In [3]:
import os
from google.colab import userdata

%cd /content/recommendation-system

token = userdata.get('GITHUB_TOKEN')
os.environ['TOKEN'] = token

!git remote set-url origin https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git

# Pull first then push
!git pull origin main --rebase
!git push origin main

print("✅ Done!")

/content/recommendation-system
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.72 KiB | 2.72 MiB/s, done.
From https://github.com/MohamedAbdeenM7/recommendation-system
 * branch            main       -> FETCH_HEAD
   bd02ee5..8bd7bf1  main       -> origin/main
CONFLICT (modify/delete): tests/test_preprocess.py deleted in HEAD and modified in b3363df (✅ M1: Add test_preprocess.py).  Version b3363df (✅ M1: Add test_preprocess.py) of tests/test_preprocess.py left in tree.
error: could not apply b3363df... ✅ M1: Add test_preprocess.py
hint: Resolve all conflicts manually, mark them as resolved with
hint: "git add/rm <conflicted_files>", then run "git rebase --continue".
hint: You can instead skip this commit: run "git rebase --skip".
hint: To abort and get back to the state before "git rebase", run "git rebas

In [4]:
import os
from google.colab import userdata

%cd /content/recommendation-system

token = userdata.get('GITHUB_TOKEN')
os.environ['TOKEN'] = token

!git remote set-url origin https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git

# Abort the rebase first
!git rebase --abort

# Force pull from remote
!git fetch origin
!git reset --hard origin/main

# Now add the file again
test_code = '''import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "src", "data")))

from preprocess import (
    remove_duplicates,
    filter_ratings,
    filter_active_users,
    filter_popular_products,
    preprocess
)

def get_sample_data():
    return pd.DataFrame({
        "user_id":    ["u1", "u1", "u1", "u1", "u1", "u2", "u2", "u2", "u2", "u2", "u1"],
        "product_id": ["p1", "p2", "p3", "p4", "p5", "p1", "p2", "p3", "p4", "p5", "p1"],
        "rating":     [5,    4,    3,    2,    1,    4,    3,    2,    1,    5,    5  ],
        "date":       pd.to_datetime(["2021-01-01"] * 11)
    })

def test_remove_duplicates():
    df = get_sample_data()
    result = remove_duplicates(df)
    assert result.duplicated().sum() == 0
    print("✅ test_remove_duplicates passed!")

def test_filter_ratings():
    df = get_sample_data()
    df.loc[0, "rating"] = 6
    df.loc[1, "rating"] = 0
    result = filter_ratings(df)
    assert result["rating"].between(1, 5).all()
    print("✅ test_filter_ratings passed!")

def test_filter_active_users():
    df = get_sample_data()
    df = remove_duplicates(df)
    result = filter_active_users(df, min_ratings=5)
    user_counts = result["user_id"].value_counts()
    assert (user_counts >= 5).all()
    print("✅ test_filter_active_users passed!")

def test_filter_popular_products():
    df = get_sample_data()
    df = remove_duplicates(df)
    result = filter_popular_products(df, min_ratings=2)
    product_counts = result["product_id"].value_counts()
    assert (product_counts >= 2).all()
    print("✅ test_filter_popular_products passed!")

def test_preprocess():
    df = get_sample_data()
    result = preprocess(df)
    assert result.duplicated().sum() == 0
    assert result["rating"].between(1, 5).all()
    assert result.isnull().sum().sum() == 0
    print("✅ test_preprocess passed!")

if __name__ == "__main__":
    print("🔄 Running all tests...\\n")
    test_remove_duplicates()
    test_filter_ratings()
    test_filter_active_users()
    test_filter_popular_products()
    test_preprocess()
    print("\\n🎉 All tests passed!")
'''

with open("tests/test_preprocess.py", "w") as f:
    f.write(test_code)
print("✅ test_preprocess.py created!")

!git config --global user.email "MohamedAbdeenM7@gmail.com"
!git config --global user.name "Student1"
!git add tests/test_preprocess.py
!git commit -m "✅ M1: Add test_preprocess.py"
!git push origin main

print("✅ test_preprocess.py pushed to GitHub!")

/content/recommendation-system
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (1/1), 907 bytes | 907.00 KiB/s, done.
From https://github.com/MohamedAbdeenM7/recommendation-system
   ebee2a3..80f65b8  student1/data-preprocessing -> origin/student1/data-preprocessing
HEAD is now at 8bd7bf1 Merge pull request #6 from MohamedAbdeenM7/student1/data-preprocessing
✅ test_preprocess.py created!
[main ed1684c] ✅ M1: Add test_preprocess.py
 1 file changed, 68 insertions(+)
 create mode 100644 tests/test_preprocess.py
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1018 bytes | 1018.00 KiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Mohame

In [5]:
import os
from google.colab import userdata

%cd /content/recommendation-system

token = userdata.get('GITHUB_TOKEN')
os.environ['TOKEN'] = token

eda_report = """# M1 — EDA Report
## Amazon Electronics Dataset — Ratings Only

---

## 1. Dataset Overview
| Metric | Value |
|--------|-------|
| Source | Amazon Product Reviews v2 |
| Category | Electronics |
| Type | Ratings Only |
| Columns | user_id, product_id, rating, date |

---

## 2. Data Cleaning
| Step | Description |
|------|-------------|
| Remove Duplicates | Removed duplicate rows |
| Filter Ratings | Kept ratings between 1 and 5 |
| Active Users | Kept users with at least 5 ratings |
| Popular Products | Kept products with at least 5 ratings |

---

## 3. EDA Insights

### 3.1 Rating Distribution
- Most users tend to give high ratings (4 and 5 stars)
- Very few users give 1 or 2 star ratings

### 3.2 User Activity
- Most users have rated only a few products
- A small number of users are very active

### 3.3 Product Popularity
- Most products have very few ratings
- A small number of products are very popular

### 3.4 Ratings over Time
- Rating activity has increased over the years
- Peak activity was observed in recent years

---

## 4. Sparsity Analysis
- The user-item matrix is highly sparse
- Most users have not rated most products
- Sparsity is above 99%

---

## 5. Output Files
| File | Description |
|------|-------------|
| ratings_clean.csv | Cleaned ratings dataset |
| eda_report.png | EDA visualizations |

---

## 6. Next Steps
- Use cleaned data for model development (M2)
- Build collaborative filtering model
- Build content-based filtering model
"""

with open("docs/M1_EDA_Report.md", "w") as f:
    f.write(eda_report)
print("✅ M1_EDA_Report.md created!")

!git config --global user.email "MohamedAbdeenM7@gmail.com"
!git config --global user.name "Student1"
!git remote set-url origin https://$TOKEN@github.com/MohamedAbdeenM7/recommendation-system.git
!git add docs/M1_EDA_Report.md
!git commit -m "✅ M1: Add EDA Report"
!git push origin main

print("✅ M1_EDA_Report.md pushed to GitHub!")

/content/recommendation-system
✅ M1_EDA_Report.md created!
[main 12d1267] ✅ M1: Add EDA Report
 1 file changed, 62 insertions(+), 1 deletion(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1.00 KiB | 1.00 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/MohamedAbdeenM7/recommendation-system.git
   ed1684c..12d1267  main -> main
✅ M1_EDA_Report.md pushed to GitHub!
